In [1]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/mishbhaul/arabic-hindi/test_challenge_ar_ar-to-hi.txt', sep='\t', header=None)   # use sep=',' if commas, sep=' ' if spaces
df.columns = ['ar']
df.to_csv('output.csv', index=False)
print(f"Total rows : {df.shape[0]}")
print(f"examples : {df.head(5)}")

Total rows : 1500
examples :                                                   ar
0  وكانت وسائل إعلام أميركية كشفت، أمس الاثنين، أ...
1  يعتقد الأسطورة آلان شيرر نجم نيوكاسل ومنتخب إن...
2  ومنذ إنهاء حكم حزب المحافظين الذي استمر 14 عام...
3                    الاعترافات الدولية بدولة فلسطين
4  نُشرت حالة أخرى لتمدد بطارية خاتم غالاكسي رينغ...


In [10]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# === CHANGE #1: where your adapter is saved ===
# In the same Kaggle session it's here:
ADAPTER_DIR = "/kaggle/input/models/mishbhaul/hi-ar-finetuned-nllb/transformers/default/1/nllb-lora-adapter"
# If you're in a NEW session, this folder is gone (Kaggle wipes /working).
# Point this to wherever you saved it — e.g. an attached dataset:
# ADAPTER_DIR = "/kaggle/input/my-nllb-adapter"

BASE_MODEL = "facebook/nllb-200-distilled-600M"   # leave as-is
SRC_LANG   = "hin_Deva"   # Arabic  — leave as-is
TGT_LANG   = "arb_Arab"   # Hindi   — leave as-is

# load base + adapter
base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [11]:
from tqdm import tqdm   # add this import at the top

@torch.inference_mode()
def translate(texts, batch_size=16):
    tokenizer.src_lang = SRC_LANG
    fb_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True,
                        truncation=True, max_length=128).to(device)
        gen = model.generate(**enc, forced_bos_token_id=fb_id,
                             max_length=128, num_beams=5)
        out.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return out

# **Arabic -> Hindi**

In [4]:
import pandas as pd

df = pd.read_csv("/kaggle/working/output.csv")
df["hi"] = translate(df["ar"].tolist())
df.to_csv("translations.csv", index=False)
print("Saved → translations.csv")

Translating: 100%|██████████| 94/94 [14:03<00:00,  8.97s/it]

Saved → translations.csv


In [6]:
data = pd.read_csv("/kaggle/working/translations.csv")
data

,ar,hi
0,وكانت وسائل إعلام أميركية كشفت، أمس الاثنين، أ...,"सोमवार को, अमेरिकी मीडिया ने खुलासा किया कि पे..."
1,يعتقد الأسطورة آلان شيرر نجم نيوكاسل ومنتخب إن...,दिग्गज एलन शेरर का मानना है कि न्यूकैसल स्टार ...
2,ومنذ إنهاء حكم حزب المحافظين الذي استمر 14 عام...,पिछले साल जुलाई में कंजरवेटिव पार्टी के 14 साल...
3,الاعترافات الدولية بدولة فلسطين,फिलिस्तीन राज्य की अंतर्राष्ट्रीय मान्यताएं
4,نُشرت حالة أخرى لتمدد بطارية خاتم غالاكسي رينغ...,कुछ समय पहले सोशल मीडिया पर गैलक्सी रिंग रिंग ...
...,...,...
1495,رحّب وزراء خارجية السعودية والأردن والإمارات و...,"सऊदी अरब, जॉर्डन, संयुक्त अरब अमीरात, इंडोनेशि..."
1496,وكشف النص التفصيلي الممتد على 21 صفحة، تفاصيل ...,21 पृष्ठों के विस्तृत लेख में अंतर्राष्ट्रीय स...
1497,الناخب العراقي اليوم أكثر وعياً ونقداً، وقد خب...,"इराकी मतदाता आज अधिक जागरूक और महत्वपूर्ण हैं,..."
1498,وبطولة أبطال الدوري في آسيا 2 تغيّرت بوجود الن...,एशिया 2 चैंपियंस लीग चैंपियनशिप इस सीज़न के अप...


In [7]:
data.to_csv('ar_hi.txt', sep='\t', index=False)

# **Hindi -> Arabic**

In [8]:
df = pd.read_csv('/kaggle/input/datasets/mishbhaul/hindi-to-arabic/test_challenge_hi_hi-to-ar.txt', sep='\t', header=None)
df.columns = ['hi']
df.to_csv('hindi.csv', index=False)
print(f"Total rows : {df.shape[0]}")
print(f"examples : {df.head(5)}")

Total rows : 1500
examples :                                                   hi
0  अध्ययन के निष्कर्ष आशाजनक रहे हैं, लेकिन वैज्ञ...
1  ऐसा लगता है कि शायद Huawei के नए इनोवेशन के जर...
2  संघीय सांख्यिकी कार्यालय के आंकड़ों के अनुसार,...
3  2011 में, युआन, जिन्हें शुरू में आगमन पर धाराप...
4  उन्होंने आगे कहा कि अंतर्राष्ट्रीय गारंटी की आ...


In [12]:
import pandas as pd

df = pd.read_csv("/kaggle/working/hindi.csv")
df["ar"] = translate(df["hi"].tolist())
df.to_csv("hi_ar.csv", index=False)
print("Saved → hi_ar.csv")

Translating: 100%|██████████| 94/94 [15:17<00:00,  9.77s/it]

Saved → hi_ar.csv


In [13]:
df.to_csv('hi_ar.txt', sep='\t', index=False)